# CRNN: CNN → BiLSTM → CTC trên IIIT5K
Notebook khám phá dữ liệu, xem luồng tensor và đọc kết quả thật. Chọn Python của `.venv` làm kernel. Nếu thiếu kernel, cài `ipykernel` trong môi trường này. Chạy các cell từ trên xuống.
Tài liệu: [cơ sở lý thuyết](../docs/co_so_ly_thuyet.md), [giao thức](../docs/thiet_ke_thuc_nghiem.md).

In [ ]:
from pathlib import Path
import os, sys, csv, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Mở notebook từ thư mục dự án.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Dữ liệu và độ dài
Chạy `python -m crnn.download` và `python -m crnn.prepare` trước. Độ dài tính trên nhãn thật đã chuyển chữ thường. Không dùng test để chọn mô hình hoặc ngưỡng mờ.

In [ ]:
from crnn.data import read_manifest
import matplotlib.pyplot as plt
splits = {s: read_manifest(ROOT / f'data/manifests/{s}.csv') for s in ['train','val','test']}
print({s: len(rows) for s, rows in splits.items()})
print(json.loads((ROOT / 'data/manifests/split_metadata.json').read_text(encoding='utf-8')))
plt.hist([len(r['label']) for r in splits['train']], bins=range(1, 25), align='left')
plt.xlabel('Ground-truth length (train)'); plt.ylabel('Number of words'); plt.show()

## Luồng tensor và CTC
CNN trả đặc trưng cao 1, mỗi cột là một bước thời gian. BiLSTM đọc hai hướng; CTC học căn chỉnh khi chỉ có nhãn từ. Với `book`, cần tối thiểu 5 bước vì hai `o` liền nhau cần blank ở giữa. Gộp lặp trước, bỏ blank sau.

In [ ]:
from crnn.model import CRNN
from crnn.codec import Alphabet, minimum_ctc_steps
from crnn.data import WordDataset, collate_words
alphabet = Alphabet()
model = CRNN(alphabet.num_classes).eval()
batch = collate_words([WordDataset('data/manifests/train.csv')[i] for i in range(2)])
with torch.no_grad(): log_probs, lengths = model(batch['images'], batch['widths'])
print('Images:', tuple(batch['images'].shape), 'Log probabilities:', tuple(log_probs.shape))
print('Valid time lengths:', lengths.tolist(), 'Targets:', batch['labels'])
print('Minimum CTC steps for book:', minimum_ctc_steps('book'))
print('Parameters:', sum(p.numel() for p in model.parameters()))
del model

## Huấn luyện
Cell dưới chỉ chạy khi đặt `RUN_TRAINING=True`. Đổi tên run khi chạy lại. Epoch và batch là cấu hình, chưa phải bằng chứng hội tụ. Bộ chọn checkpoint dùng validation accuracy, rồi loss thấp hơn khi hòa.

In [ ]:
import subprocess
RUN_TRAINING = False
RUN = ROOT / 'runs/baseline_seed42'
if RUN_TRAINING:
    subprocess.run([sys.executable, '-m', 'crnn.train', '--train', 'data/manifests/train.csv', '--val', 'data/manifests/val.csv', '--output', str(RUN), '--epochs', '50', '--batch-size', '16', '--seed', '42', '--device', 'cuda'], check=True)

## Đường học và báo cáo
Chạy đánh giá/phân tích theo README trước khi đọc kết quả. Nếu run chưa có kết quả, notebook thông báo thiếu tệp và không tạo số liệu minh họa.

In [ ]:
history_path = RUN / 'history.csv'
if history_path.exists():
    with history_path.open(encoding='utf-8', newline='') as f: history = list(csv.DictReader(f))
    epochs = [int(r['epoch']) for r in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split in ['train', 'val']:
        axes[0].plot(epochs, [float(r[f'{split}_loss']) for r in history], label=split)
        axes[1].plot(epochs, [100 * float(r[f'{split}_word_accuracy']) for r in history], label=split)
    axes[0].set_title('CTC loss'); axes[1].set_title('Word accuracy (%)')
    for ax in axes: ax.set_xlabel('Epoch'); ax.legend()
    plt.show()
else: print('Chưa có history.csv cho run đã chọn.')
summary_path = RUN / 'analysis/summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary['overall'], indent=2))
    print('Đọc', RUN / 'analysis/report.md')
else: print('Chưa có phân tích cho run đã chọn.')

In [ ]:
from PIL import Image
errors_path = RUN / 'analysis/errors.csv'
if errors_path.exists():
    with errors_path.open(encoding='utf-8', newline='') as f: errors = list(csv.DictReader(f))[:8]
    if errors:
        fig, axes = plt.subplots(4, 2, figsize=(12, 10))
        for ax in axes.flat: ax.axis('off')
        for ax, row in zip(axes.flat, errors):
            ax.imshow(Image.open(row['image_path']))
            ax.set_title(f"{row['id']} | GT: {row['label']} | Pred: {row['prediction']}", fontsize=9)
        plt.tight_layout(); plt.show()
    else: print('Không có lỗi trong tệp đã chọn.')
else: print('Chưa có errors.csv.')